# Fixes for static files

* [#706](https://github.com/astronomy-commons/hats-import/issues/706) Make summary generation more flexible
* [#744](https://github.com/astronomy-commons/hats-import/issues/744) Create static files by default

In [50]:
import tempfile
from pathlib import Path

from dask.distributed import Client

from hats_import import pipeline_with_client, ImportArguments, CollectionArguments

In [51]:
tmp_path = tempfile.TemporaryDirectory()
tmp_dir = tmp_path.name

hats_import_dir = "."
client = Client(n_workers=1, threads_per_worker=1, local_directory=tmp_dir)

/home/delucchi/.virtualenvs/oct3/lib/python3.14/site-packages/distributed/node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41443 instead
  warnings.warn(


In [52]:
args = ImportArguments(
    output_artifact_name="small_sky_demo_catalog",
    input_path=Path(hats_import_dir) / "small_sky",
    file_reader="csv",
    output_path=".",
    highest_healpix_order=2,
    create_summary_md_kwargs={"huggingface_metadata": True},
)

pipeline_with_client(args, client)


Catalog: Planning  :   0%|          | 0/4 [00:00<?, ?it/s]

Catalog: Mapping   :   0%|          | 0/1 [00:00<?, ?it/s]

Catalog: Binning   :   0%|          | 0/2 [00:00<?, ?it/s]

Catalog: Splitting :   0%|          | 0/1 [00:00<?, ?it/s]

Catalog: Reducing  :   0%|          | 0/1 [00:00<?, ?it/s]

Catalog: Finishing :   0%|          | 0/10 [00:00<?, ?it/s]

In [53]:
!ls small_sky_demo_catalog

README.md	 partition_info.csv		   properties
dataset		 partition_info.png		   skymap.fits
hats.properties	 per_partition_statistics.parquet  skymap.png
index.html	 point_map.fits


See that we have two pngs there:

* `partition_info.png`
* `skymap.png`

In [54]:
!head small_sky_demo_catalog/README.md


---
configs:
- config_name: default
  data_dir: dataset
tags:
- astronomy
---

# small_sky_demo_catalog HATS Catalog 


And see the hugging face metadata header in the readme? NICE!

Let's repeat, but for a collection. 

In [55]:
args = (
    CollectionArguments(
        output_artifact_name="small_sky_demo_collection",
        output_path=".",
        progress_bar=False,
        create_summary_md_kwargs={"huggingface_metadata": True},
    )
    .catalog(
        input_path=Path(hats_import_dir) / "small_sky",
        file_reader="csv",
        highest_healpix_order=2,
    )
    .add_margin(margin_threshold=3600.0)
    .add_margin(margin_threshold=5.0)
    .add_index(indexing_column="id", include_healpix_29=False)
)
pipeline_with_client(args, client)

/home/delucchi/git/summary/hats/src/hats/catalog/partition_info.py:113: UserWarning: Computing partitions from catalog parquet files. This may be slow.
  warnings.warn("Computing partitions from catalog parquet files. This may be slow.")
/home/delucchi/git/summary/hats-import/src/hats_import/index/arguments.py:101: UserWarning: Index tables are not pixel-partitioned - ignoring argument(s) create_per_partition_stats.
  warnings.warn(


In [56]:
!ls small_sky_demo_collection

README.md	       small_sky_demo_collection
collection.properties  small_sky_demo_collection_1deg
index.html	       small_sky_demo_collection_5arcs
partition_info.png     small_sky_demo_collection_id
skymap.png


In [57]:
!tree small_sky_demo_collection/small_sky_demo_collection

small_sky_demo_collection/small_sky_demo_collection
├── dataset
│   ├── _common_metadata
│   ├── _metadata
│   └── Norder=0
│       └── Dir=0
│           └── Npix=11.parquet
├── hats.properties
├── index.html
├── partition_info.csv
├── partition_info.png
├── per_partition_statistics.parquet
├── point_map.fits
├── properties
├── README.md
├── skymap.fits
└── skymap.png

4 directories, 13 files


In [58]:
!head small_sky_demo_collection/small_sky_demo_collection/README.md


---
configs:
- config_name: default
  data_dir: dataset
tags:
- astronomy
---

# small_sky_demo_collection HATS Catalog 


We shouldn't see the stored images, because the margin is empty.

In [59]:
!tree small_sky_demo_collection/small_sky_demo_collection_5arcs

small_sky_demo_collection/small_sky_demo_collection_5arcs
├── dataset
│   ├── _common_metadata
│   └── _metadata
├── hats.properties
├── index.html
├── partition_info.csv
├── properties
└── README.md

2 directories, 7 files


BUT, where there are contents to the margin, we get a `partition_info.png`

In [60]:
!tree small_sky_demo_collection/small_sky_demo_collection_1deg

small_sky_demo_collection/small_sky_demo_collection_1deg
├── dataset
│   ├── _common_metadata
│   ├── _metadata
│   └── Norder=0
│       └── Dir=0
│           └── Npix=4.parquet
├── hats.properties
├── index.html
├── partition_info.csv
├── partition_info.png
├── per_partition_statistics.parquet
├── properties
└── README.md

4 directories, 10 files


In [61]:
!head small_sky_demo_collection/small_sky_demo_collection_5arcs/README.md


---
configs:
- config_name: default
  data_dir: dataset
tags:
- astronomy
---

# small_sky_demo_collection_5arcs HATS Margin Catalog


In [62]:
!tree small_sky_demo_collection/small_sky_demo_collection_id

small_sky_demo_collection/small_sky_demo_collection_id
├── dataset
│   ├── _common_metadata
│   ├── index
│   │   └── part.0.parquet
│   └── _metadata
├── hats.properties
├── index.html
├── properties
└── README.md

3 directories, 7 files


In [63]:
!head small_sky_demo_collection/small_sky_demo_collection_id/README.md


---
configs:
- config_name: default
  data_dir: dataset
tags:
- astronomy
---

# small_sky_demo_collection_id HATS Index Catalog
